# LCSumoEnv — Model Evaluation

This notebook loads a trained DQN agent (lcCooperative / lcAssertive) and evaluates it on traffic data.

## Instructions
1. Place your trained `dqn_best.pt` in `models/lc/`
2. Place your evaluation CSV in `data/lc/eval_data.csv` (format: CCTV Data Remastered.csv \u2014 13 semicolon-separated columns, first column is label)
3. Adjust config variables below if needed
4. Run all cells

In [ ]:
import os
import sys
import csv
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

_proj_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
for p in [_proj_root, os.path.join(_proj_root, "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)
if "SUMO_HOME" in os.environ:
    sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))

from src.data_repo import DataRepo
from src.agents.dqn_agent import DQNAgent

%matplotlib inline

In [ ]:
from src.envs.lc_env import LCSumoEnv

MODEL_PATH = Path("models/lc/dqn_best.pt")
DATA_PATH = Path("data/lc/eval_data.csv")
SUMO_BINARY = "sumo"
WARMUP_MINUTES = 5
PERIOD_MINUTES = 5
STEP_LENGTH = 0.05
MAX_ROWS = None  # set to int to limit rows for quick testing
SAVE_RESULTS = True
USE_HOURLY = True  # True=columns 7-12 (\x6), False=columns 1-6 (10-min)

STATE_DIM = 6
ACTION_DIM = 9
ENV_CLASS = LCSumoEnv

print("Configuration:")
print(f"  Model: {MODEL_PATH}")
print(f"  Data:  {DATA_PATH}")
print(f"  Env:   LCSumoEnv (state_dim={STATE_DIM}, action_dim={ACTION_DIM})")
print(f"  step_length={STEP_LENGTH}, warmup={WARMUP_MINUTES}min, period={PERIOD_MINUTES}min")
print(f"  MAX_ROWS={'unlimited' if MAX_ROWS is None else MAX_ROWS}")
print(f"  NOTE: step_length=0.05 means ~20x more sim steps than SF env. Expect slower eval.")

In [ ]:
def load_eval_data(csv_path, max_rows=None, hourly=True):
    df = pd.read_csv(csv_path, sep=";", header=0)
    cols = list(range(7, 13)) if hourly else list(range(1, 7))
    data = df.iloc[:, cols].values.astype(np.float32)
    mask = (data[:, 4] != 0) & (data[:, 5] != 0)
    data = data[mask]
    if max_rows is not None:
        data = data[:max_rows]
    print(f"Loaded {len(data)} data point(s) from {csv_path} ({"hourly" if hourly else "10-min"})")
    for i, row in enumerate(data):
        print(f"  [{i}] SL={row[0]:.0f}  SPT={row[1]:.0f}  UL={row[2]:.0f}  UPT={row[3]:.0f}  "
              f"OS={row[4]:.0f}  OU={row[5]:.0f}")
    return data

data = load_eval_data(DATA_PATH, MAX_ROWS, USE_HOURLY)

In [ ]:
repo = DataRepo()
env = ENV_CLASS(
    sumo_config_path=str(repo.sumo_config_path),
    sumo_binary=SUMO_BINARY,
    step_length=STEP_LENGTH,
    warmup_minutes=WARMUP_MINUTES,
    period_minutes=PERIOD_MINUTES,
    reward_scale=1.0,
    reset_params_on_reset=True,
)
env.set_target_data(data)
flow_max = float(np.max(data[:, :4]))
print(f"flow_max = {flow_max:.1f}  (used for state normalization)")

In [ ]:
agent = DQNAgent(state_dim=STATE_DIM, action_dim=ACTION_DIM)
agent.load(str(MODEL_PATH))
agent.q_net.eval()
agent.target_net.eval()
print(f"Model loaded from {MODEL_PATH.resolve()}")
print(f"Device: {agent.device}")

In [ ]:
results = []
n_data = len(data)
param_names = list(env.action_map[0].keys())

print(f"Running evaluation on {n_data} data point(s)...")
print(f"(4 steps per data point, epsilon=0 greedy policy)")
print(f"(CAUTION: step_length={STEP_LENGTH} \u2192 ~{int(60/STEP_LENGTH * PERIOD_MINUTES)} sim steps per step)\n")

for data_idx in range(n_data):
    obs, _ = env.reset()
    row_steps = []
    for step in range(4):
        action = agent.act(obs, epsilon=0.0)
        next_obs, reward, terminated, _, info = env.step(action)
        entry = {"step": step + 1, "action": int(action)}
        entry.update(info["params"])
        entry.update({
            "sim_north": info["sim_north"],
            "sim_south": info["sim_south"],
            "expected_north": info["expected_north"],
            "expected_south": info["expected_south"],
            "reward": float(reward),
        })
        row_steps.append(entry)
        obs = next_obs

    first_mape = -row_steps[0]["reward"]
    last_mape = -row_steps[-1]["reward"]
    flow_str = (f"SL={data[data_idx][0]:.0f} SPT={data[data_idx][1]:.0f} "
                f"UL={data[data_idx][2]:.0f} UPT={data[data_idx][3]:.0f}")
    params_str = ", ".join(f"{k}={row_steps[-1][k]:.2f}" for k in param_names)
    print(f"  [{data_idx+1:2d}/{n_data}] {flow_str}  "
          f"MAPE: {first_mape:.3f} \u2192 {last_mape:.3f}  ({params_str})")

    results.append({
        "data_idx": data_idx,
        "sl": float(data[data_idx][0]),
        "spt": float(data[data_idx][1]),
        "ul": float(data[data_idx][2]),
        "upt": float(data[data_idx][3]),
        "os": float(data[data_idx][4]),
        "ou": float(data[data_idx][5]),
        "steps": row_steps,
    })

print(f"\nDone \u2014 {n_data} data point(s) evaluated.")

In [ ]:
rows_summary = []
for r in results:
    first = r["steps"][0]
    last = r["steps"][-1]
    mape_start = -first["reward"]
    mape_end = -last["reward"]
    row = {
        "DataIdx": r["data_idx"],
        "SL": r["sl"], "SPT": r["spt"], "UL": r["ul"], "UPT": r["upt"],
        "OS": r["os"], "OU": r["ou"],
    }
    for k in param_names:
        row[f"{k}_start"] = first[k]
        row[f"{k}_end"] = last[k]
    row.update({
        "SimN_start": first["sim_north"], "SimS_start": first["sim_south"],
        "SimN_end": last["sim_north"], "SimS_end": last["sim_south"],
        "ExpN": first["expected_north"], "ExpS": first["expected_south"],
        "MAPE_start": round(mape_start, 4),
        "MAPE_end": round(mape_end, 4),
        "Improvement": round(mape_start - mape_end, 4),
    })
    rows_summary.append(row)

df_summary = pd.DataFrame(rows_summary)
print("Per-Data-Point Summary:")
display(df_summary)

print("\nAggregate Metrics:")
avg_start = df_summary["MAPE_start"].mean()
avg_end = df_summary["MAPE_end"].mean()
print(f"  Mean MAPE (step 1): {avg_start:.4f}")
print(f"  Mean MAPE (step 4): {avg_end:.4f}")
print(f"  Mean improvement:   {(avg_start - avg_end):+.4f}")
print(f"  Rows improved:      {(df_summary['Improvement'] > 0).sum()} / {len(df_summary)}")
print(f"  Rows worsened:      {(df_summary['Improvement'] < 0).sum()} / {len(df_summary)}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1 — Sim vs Expected North
ax = axes[0, 0]
ax.scatter(df_summary["ExpN"], df_summary["SimN_end"], alpha=0.7, s=30)
lo = min(df_summary["ExpN"].min(), df_summary["SimN_end"].min())
hi = max(df_summary["ExpN"].max(), df_summary["SimN_end"].max())
ax.plot([lo, hi], [lo, hi], "r--", alpha=0.4, linewidth=1)
ax.set_xlabel("Expected North (hourly)"); ax.set_ylabel("Sim North (final)")
ax.set_title("Sim vs Expected \u2014 Northbound"); ax.grid(True, alpha=0.3)
ax.set_aspect("equal")

# 2 — Sim vs Expected South
ax = axes[0, 1]
ax.scatter(df_summary["ExpS"], df_summary["SimS_end"], alpha=0.7, s=30)
lo = min(df_summary["ExpS"].min(), df_summary["SimS_end"].min())
hi = max(df_summary["ExpS"].max(), df_summary["SimS_end"].max())
ax.plot([lo, hi], [lo, hi], "r--", alpha=0.4, linewidth=1)
ax.set_xlabel("Expected South (hourly)"); ax.set_ylabel("Sim South (final)")
ax.set_title("Sim vs Expected \u2014 Southbound"); ax.grid(True, alpha=0.3)
ax.set_aspect("equal")

# 3 — MAPE: Step 1 vs Step 4
ax = axes[0, 2]
x = range(len(df_summary))
width = 0.35
ax.bar([i - width/2 for i in x], df_summary["MAPE_start"], width, label="Step 1", alpha=0.7)
ax.bar([i + width/2 for i in x], df_summary["MAPE_end"], width, label="Step 4", alpha=0.7)
ax.set_xlabel("Data Point"); ax.set_ylabel("MAPE")
ax.set_title("MAPE: Step 1 vs Step 4"); ax.legend(); ax.grid(True, alpha=0.3)

# 4 — Final parameters per data point
ax = axes[1, 0]
colors = ["tab:blue", "tab:orange"]
for i, k in enumerate(param_names):
    ax.plot(x, df_summary[f"{k}_end"], "o-", color=colors[i], label=k)
ax.set_xlabel("Data Point"); ax.set_ylabel("Value")
ax.set_title("Final Parameters per Data Point")
ax.legend(); ax.grid(True, alpha=0.3)

# 5 — Improvement histogram
ax = axes[1, 1]
bins = max(5, min(20, len(df_summary) // 3))
ax.hist(df_summary["Improvement"], bins=bins, alpha=0.7, edgecolor="black")
ax.axvline(0, color="r", linestyle="--", alpha=0.5)
ax.set_xlabel("MAPE Improvement"); ax.set_ylabel("Count")
ax.set_title("Distribution of Improvement"); ax.grid(True, alpha=0.3)

# 6 — Reward trajectory (first 5 rows)
ax = axes[1, 2]
for r in results[:5]:
    rewards = [s["reward"] for s in r["steps"]]
    ax.plot(range(1, 5), rewards, "o-", label=f"Row {r['data_idx']}")
ax.set_xlabel("Step"); ax.set_ylabel("Reward")
ax.set_title("Reward Trajectory (first 5 rows)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if SAVE_RESULTS:
    rows_flat = []
    for r in results:
        for s in r["steps"]:
            row = {
                "data_idx": r["data_idx"],
                "sl": r["sl"], "spt": r["spt"], "ul": r["ul"], "upt": r["upt"],
                "os": r["os"], "ou": r["ou"],
                "step": s["step"], "action": s["action"],
            }
            for k in param_names:
                row[k] = s[k]
            row.update({
                "sim_north": s["sim_north"], "sim_south": s["sim_south"],
                "expected_north": s["expected_north"], "expected_south": s["expected_south"],
                "reward": s["reward"],
            })
            rows_flat.append(row)
    df_flat = pd.DataFrame(rows_flat)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path("data/lc")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"eval_results_{ts}.csv"
    df_flat.to_csv(out_path, index=False)
    print(f"Results saved to {out_path}")
else:
    print("SAVE_RESULTS is False, skipping CSV export.")

In [ ]:
env.close()
print("SUMO processes cleaned up.")